In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree,export_text
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import RFE
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA


OPIS SKUPA PODATAKA

The dataset consists of 10 numerical and 8 categorical attributes.
The 'Revenue' attribute can be used as the class label.

"Administrative", "Administrative Duration", "Informational", "Informational Duration", "Product Related" and "Product Related Duration" represent the number of different types of pages visited by the visitor in that session and total time spent in each of these page categories. The values of these features are derived from the URL information of the pages visited by the user and updated in real time when a user takes an action, e.g. moving from one page to another. The "Bounce Rate", "Exit Rate" and "Page Value" features represent the metrics measured by "Google Analytics" for each page in the e-commerce site. The value of "Bounce Rate" feature for a web page refers to the percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. The value of "Exit Rate" feature for a specific web page is calculated as for all pageviews to the page, the percentage that were the last in the session. The "Page Value" feature represents the average value for a web page that a user visited before completing an e-commerce transaction. The "Special Day" feature indicates the closeness of the site visiting time to a specific special day (e.g. Mother’s Day, Valentine's Day) in which the sessions are more likely to be finalized with transaction. The value of this attribute is determined by considering the dynamics of e-commerce such as the duration between the order date and delivery date. For example, for Valentina’s day, this value takes a nonzero value between February 2 and February 12, zero before and after this date unless it is close to another special day, and its maximum value of 1 on February 8. The dataset also includes operating system, browser, region, traffic type, visitor type as returning or new visitor, a Boolean value indicating whether the date of the visit is weekend, and month of the year.

In [23]:
def metrika(title,preds,y_test):
    print(title, accuracy_score(y_test,preds))
    # print("Confusion Matrix:")
    # print(confusion_matrix(y_test, preds))
    # print("\nClassification Report:")
    # print(classification_report(y_test, preds))
    print("-------------------------------------------------------------------")

In [24]:
def predicting(model, title, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrika(title ,preds ,y_test)

In [25]:
df = pd.read_csv(r'podaci\online+shoppers+purchasing+intention+dataset\online_shoppers_intention preprocessed.csv', encoding='cp1252', sep=',')
print(df.columns)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')


In [26]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,2,1,0


In [27]:
X = df.copy(deep=True)

In [28]:
X = X.drop("Revenue", axis=1)
y = df["Revenue"]

In [29]:
engagement_features = ["Administrative","Administrative_Duration","Informational","Informational_Duration","ProductRelated",
        "ProductRelated_Duration","BounceRates","ExitRates","PageValues"]

technical_features = ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]

time_features = ["SpecialDay","Month","Weekend"]

feature_names = X.columns

In [30]:
numeric_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues"
]

In [31]:
pca = PCA(n_components=2)
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

In [32]:
X_log = X.copy(deep=True)

In [33]:
X_log[numeric_features] = np.log1p(X_log[numeric_features])     # Logaritamsko skaliranje
X_sca_std = scaler_standard.fit_transform(X)                    # Standard scaler
X_sca_mm = scaler_minmax.fit_transform(X)                       # MinMax scaler
X_pca = pca.fit_transform(X)                                    # Samo PCA

X_pca_sca_std = pca.fit_transform(X_sca_std)                    # Standard scaler + PCA
X_pca_sca_mm = pca.fit_transform(X_sca_mm)                      # MinMax scaler + PCA


X_eng_pca = pca.fit_transform(X[engagement_features])
X_tec_pca = pca.fit_transform(X[technical_features])
X_time_pca = pca.fit_transform(X[time_features])

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123, stratify=y)

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std = train_test_split(X_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm = train_test_split(X_sca_mm, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=123, stratify=y)

X_train_pca_sca_std, X_test_pca_sca_std, y_train_pca_sca_std, y_test_pca_sca_std = train_test_split(X_pca_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca_sca_mm, X_test_pca_sca_mm, y_train_pca_sca_mm, y_test_pca_sca_mm = train_test_split(X_pca_sca_mm, y, test_size=0.2, random_state=123, stratify=y)


X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(X_eng_pca, y, test_size=0.2, random_state=123, stratify=y)
X_train_tec, X_test_tec, y_train_tec, y_test_tec = train_test_split(X_tec_pca, y, test_size=0.2, random_state=123, stratify=y)
X_train_time, X_test_time, y_train_time, y_test_time = train_test_split(X_time_pca, y, test_size=0.2, random_state=123, stratify=y)

In [ ]:
datasets = [
    ("Originalni podaci", X_train, X_test, y_train, y_test),
    ("Log transformacija", X_train_log, X_test_log, y_train_log, y_test_log),
    ("StandardScaler", X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std),
    ("MinMaxScaler", X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm),
    ("PCA", X_train_pca, X_test_pca, y_train_pca, y_test_pca),
    ("StandardScaler + PCA", X_train_pca_sca_std, X_test_pca_sca_std,
     y_train_pca_sca_std, y_test_pca_sca_std),
    (" MinMaxScaler + PCA", X_train_pca_sca_mm, X_test_pca_sca_mm,
     y_train_pca_sca_mm, y_test_pca_sca_mm),
    ("Engineering PCA", X_train_eng, X_test_eng, y_train_eng, y_test_eng),
    ("Technical PCA", X_train_tec, X_test_tec, y_train_tec, y_test_tec),
    ("Time PCA", X_train_time, X_test_time, y_train_time, y_test_time)
]

PREDVIDJANJE

In [36]:
dt = DecisionTreeClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(dt, f"Stablo odlucivanja + {name}", X_tr, y_tr, X_te, y_te)

Stablo odlucivanja + Originalni podaci 0.8572587185725872
-------------------------------------------------------------------
Stablo odlucivanja + Log transformacija 0.8564476885644768
-------------------------------------------------------------------
Stablo odlucivanja + StandardScaler 0.8532035685320357
-------------------------------------------------------------------
Stablo odlucivanja + MinMaxScaler 0.8564476885644768
-------------------------------------------------------------------
Stablo odlucivanja + PCA 0.7684509326845094
-------------------------------------------------------------------
Stablo odlucivanja + PCA + StandardScaler 0.7927818329278183
-------------------------------------------------------------------
Stablo odlucivanja + PCA + MinMaxScaler 0.7660178426601785
-------------------------------------------------------------------
Stablo odlucivanja + Engineering PCA 0.7708840227088403
-------------------------------------------------------------------
Stablo odlu

In [37]:
rf = RandomForestClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(rf, f"Random Forest + {name}", X_tr, y_tr, X_te, y_te)

Random Forest + Originalni podaci 0.9087591240875912
-------------------------------------------------------------------
Random Forest + Log transformacija 0.9075425790754258
-------------------------------------------------------------------
Random Forest + StandardScaler 0.9055150040551501
-------------------------------------------------------------------
Random Forest + MinMaxScaler 0.9030819140308192
-------------------------------------------------------------------
Random Forest + PCA 0.8256285482562855
-------------------------------------------------------------------
Random Forest + PCA + StandardScaler 0.8446877534468775
-------------------------------------------------------------------
Random Forest + PCA + MinMaxScaler 0.8195458231954582
-------------------------------------------------------------------
Random Forest + Engineering PCA 0.8199513381995134
-------------------------------------------------------------------
Random Forest + Technical PCA 0.8333333333333334
--

In [38]:
mlp = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(mlp, f"MLP + {name}", X_tr, y_tr, X_te, y_te)

MLP + Originalni podaci 0.8811841038118411
-------------------------------------------------------------------
MLP + Log transformacija 0.9014598540145985
-------------------------------------------------------------------
MLP + StandardScaler 0.8969991889699919
-------------------------------------------------------------------
MLP + MinMaxScaler 0.9018653690186537
-------------------------------------------------------------------
MLP + PCA 0.8398215733982157
-------------------------------------------------------------------
MLP + PCA + StandardScaler 0.856853203568532
-------------------------------------------------------------------
MLP + PCA + MinMaxScaler 0.8450932684509327
-------------------------------------------------------------------
MLP + Engineering PCA 0.8446877534468775
-------------------------------------------------------------------
MLP + Technical PCA 0.8450932684509327
-------------------------------------------------------------------
MLP + Time PCA 0.84509326

In [39]:
nb = GaussianNB()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(nb, f"NB + {name}", X_tr, y_tr, X_te, y_te)

NB + Originalni podaci 0.8467153284671532
-------------------------------------------------------------------
NB + Log transformacija 0.7084347120843472
-------------------------------------------------------------------
NB + StandardScaler 0.7931873479318735
-------------------------------------------------------------------
NB + MinMaxScaler 0.7931873479318735
-------------------------------------------------------------------
NB + PCA 0.8272506082725061
-------------------------------------------------------------------
NB + PCA + StandardScaler 0.8454987834549879
-------------------------------------------------------------------
NB + PCA + MinMaxScaler 0.8450932684509327
-------------------------------------------------------------------
NB + Engineering PCA 0.8272506082725061
-------------------------------------------------------------------
NB + Technical PCA 0.8450932684509327
-------------------------------------------------------------------
NB + Time PCA 0.8450932684509327


In [40]:
knn = KNeighborsClassifier(n_neighbors=3)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(knn, f"KNN + {name}", X_tr, y_tr, X_te, y_te)

KNN + Originalni podaci 0.8540145985401459
-------------------------------------------------------------------
KNN + Log transformacija 0.878345498783455
-------------------------------------------------------------------
KNN + StandardScaler 0.8698296836982968
-------------------------------------------------------------------
KNN + MinMaxScaler 0.845904298459043
-------------------------------------------------------------------
KNN + PCA 0.8021086780210868
-------------------------------------------------------------------
KNN + PCA + StandardScaler 0.8304947283049473
-------------------------------------------------------------------
KNN + PCA + MinMaxScaler 0.8049472830494728
-------------------------------------------------------------------
KNN + Engineering PCA 0.8021086780210868
-------------------------------------------------------------------
KNN + Technical PCA 0.8098134630981346
-------------------------------------------------------------------
KNN + Time PCA 0.758718572

In [41]:
lr = LogisticRegression()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(lr, f"lr + {name}", X_tr, y_tr, X_te, y_te)

lr + Originalni podaci 0.884022708840227
-------------------------------------------------------------------
lr + Log transformacija 0.8986212489862125
-------------------------------------------------------------------
lr + StandardScaler 0.8860502838605029
-------------------------------------------------------------------
lr + MinMaxScaler 0.8791565287915653
-------------------------------------------------------------------
lr + PCA 0.8446877534468775
-------------------------------------------------------------------
lr + PCA + StandardScaler 0.8544201135442011
-------------------------------------------------------------------
lr + PCA + MinMaxScaler 0.8450932684509327
-------------------------------------------------------------------
lr + Engineering PCA 0.8446877534468775
-------------------------------------------------------------------
lr + Technical PCA 0.8450932684509327
-------------------------------------------------------------------
lr + Time PCA 0.8450932684509327
-

c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.

In [42]:
svm = SVC(kernel='rbf', probability=True)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(svm, f"SVM + {name}", X_tr, y_tr, X_te, y_te)

SVM + Originalni podaci 0.8475263584752636
-------------------------------------------------------------------
SVM + Log transformacija 0.8925385239253852
-------------------------------------------------------------------
SVM + StandardScaler 0.8933495539334956
-------------------------------------------------------------------
SVM + MinMaxScaler 0.8815896188158961
-------------------------------------------------------------------
SVM + PCA 0.8450932684509327
-------------------------------------------------------------------
SVM + PCA + StandardScaler 0.8548256285482563
-------------------------------------------------------------------
SVM + PCA + MinMaxScaler 0.8450932684509327
-------------------------------------------------------------------
SVM + Engineering PCA 0.8450932684509327
-------------------------------------------------------------------
SVM + Technical PCA 0.8450932684509327
-------------------------------------------------------------------
SVM + Time PCA 0.8450932

In [43]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(xgb, f"XGB + {name}", X_tr, y_tr, X_te, y_te)

XGB + Originalni podaci 0.902676399026764
-------------------------------------------------------------------
XGB + Log transformacija 0.902676399026764
-------------------------------------------------------------------


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGB + StandardScaler 0.902676399026764
-------------------------------------------------------------------
XGB + MinMaxScaler 0.902676399026764
-------------------------------------------------------------------
XGB + PCA 0.8292781832927818
-------------------------------------------------------------------
XGB + PCA + StandardScaler 0.8507704785077048
-------------------------------------------------------------------


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNI

XGB + PCA + MinMaxScaler 0.8418491484184915
-------------------------------------------------------------------
XGB + Engineering PCA 0.829683698296837
-------------------------------------------------------------------
XGB + Technical PCA 0.8373884833738848
-------------------------------------------------------------------
XGB + Time PCA 0.8450932684509327
-------------------------------------------------------------------


c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [44]:
y_train_oh = to_categorical(y_train)
y_test_oh = to_categorical(y_test)

model = Sequential([
    Dense(64, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train_oh, epochs=50, batch_size=8, verbose=0)
_, acc = model.evaluate(X_test, y_test_oh, verbose=0)
print("Deep Learning Accuracy:", acc)

# feature_selection(model, X_train, y_train)

c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Deep Learning Accuracy: 0.886455774307251
